In [1]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation
from sklearn.metrics import jaccard_score
import matplotlib.pyplot as plt

d:\PROJ\vesselmonitoring\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
IMG_SIZE = 512
BATCH_SIZE = 4  
EPOCHS = 5
LR = 6e-5

IMAGE_DIR = "data/train/images/"
MASK_DIR  = "data/train/labels/"

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# ======================
# COLOR MAPPING
# ======================
COLOR_MAPPING = {
    (0, 0, 0): 0,
    (0, 255, 255): 1,
    (255, 0, 0): 2,
    (153, 76, 0): 3,
    (0, 153, 0): 4
}

def rgb_to_class(mask):
    mask = mask.astype(np.uint8)
    class_mask = np.zeros(mask.shape[:2], dtype=np.int64)

    for rgb, cls in COLOR_MAPPING.items():
        matches = np.all(mask == rgb, axis=-1)
        class_mask[matches] = cls

    return class_mask


In [5]:

# ======================
# AUGMENTATIONS
# ======================
def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.HueSaturationValue(p=0.3),
        A.Normalize(mean=(0.485,0.456,0.406),
                    std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406),
                    std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])

In [6]:
# ======================
# DATASET
# ======================
class OilDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transforms=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms

        self.images = sorted(os.listdir(img_dir))
        self.masks  = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = np.array(Image.open(os.path.join(self.img_dir, self.images[idx])).convert("RGB"))
        mask = np.array(Image.open(os.path.join(self.mask_dir, self.masks[idx])).convert("RGB"))

        mask = rgb_to_class(mask)

        if self.transforms:
            aug = self.transforms(image=img, mask=mask)
            img = aug["image"]
            mask = aug["mask"]

        return img, mask.long()

In [13]:

# ======================
# SPLIT
# ======================
full_dataset = OilDataset(IMAGE_DIR, MASK_DIR, transforms=None)

train_size = int(0.7 * len(full_dataset))
val_size   = int(0.15 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

train_ds.dataset.transforms = get_train_transforms()
val_ds.dataset.transforms   = get_val_transforms()
test_ds.dataset.transforms  = get_val_transforms()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False )
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [14]:
# ======================
# MODEL (SegFormer B5)
# ======================
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640",
    num_labels=5,
    ignore_mismatched_sizes=True
)

model.to(DEVICE)

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `150`.
Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 13856.48it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b5-finetuned-ade-640-640
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([5, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([5])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [15]:

# ======================
# LOSS (Dice + Focal)
# ======================
def dice_loss(pred, target, smooth=1e-6):
    pred = torch.softmax(pred, dim=1)

    total_loss = 0
    for cls in range(1, 5):  # ignore background
        pred_cls = pred[:, cls]
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        loss = 1 - (2 * intersection + smooth) / (union + smooth)
        total_loss += loss

    return total_loss / 4


focal_loss = nn.CrossEntropyLoss()  # simplified

def loss_fn(pred, target):
    return focal_loss(pred, target) + dice_loss(pred, target)

In [16]:
# ======================
# OPTIMIZER
# ======================
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

In [19]:
# ======================
# TRAINING
# ======================
def train_epoch(loader):
    model.train()
    total_loss = 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(pixel_values=imgs)
        logits = outputs.logits

        logits = nn.functional.interpolate(
            logits,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def val_epoch(loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

            outputs = model(pixel_values=imgs)
            logits = outputs.logits

            logits = nn.functional.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            loss = loss_fn(logits, masks)
            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# ======================
# TRAIN LOOP
# ======================
best_val = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_loss = val_epoch(val_loader)

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}: Train {train_loss:.4f} | Val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_segformer.pth")

In [ ]:

# ======================
# EVALUATION
# ======================
def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)

            outputs = model(pixel_values=imgs)
            logits = outputs.logits

            logits = nn.functional.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            labels = masks.numpy()

            preds_all.extend(preds.flatten())
            labels_all.extend(labels.flatten())

    iou = jaccard_score(labels_all, preds_all, average='macro')
    print("Mean IoU:", iou)

evaluate(test_loader)

In [ ]:
# ======================
# VISUALIZATION
# ======================
def visualize(img, pred):
    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(img)
    plt.title("Image")

    plt.subplot(1,2,2)
    plt.imshow(pred, cmap='jet')
    plt.title("Prediction")

    plt.show()